# 🔒 Maya's Model Sits a Fair Exam: Day 2 Practical
### Week 3 · Cross-Validation and ML Pipelines

Maya's model scored 75% on Tuesday. Dr. Kofi wants proof before he spends the phone budget on it.
Today you'll **test a model honestly, pack the preparation into a pipeline, and build Maya's call list**.

**How to use this notebook**
- Keep this notebook and **followup_visits.csv** in the same folder. It's the same **raw** file as Tuesday, gaps and all.
- Press **Shift + Enter** to run a cell, and run the steps **in order**: every step reuses earlier variables.
- Compare your result with the ✅ *expected output*. A different scikit-learn version can move a score by a point or so.
- Muddled? Use **Kernel → Restart & Run All**.

> 🔒 All data in this notebook is made up.

---
## Step 1 · Lock away a test set  *(~5 min)*
**Concept:** X and y, train_test_split, stratify

📖 On Tuesday Maya's model scored well on patients it had never seen. Today she does it properly. First rule: hide some patients in a locked drawer, and don't open it until the very end.

🎯 **Your task**
1. Load the **raw** file `followup_visits.csv`. Yes, the one with the gaps.
2. Put the features in `X` and the target in `y`.
3. Split off 20% as the test set and check that both parts have a similar missed rate.

<details><summary>👀 Need the code? Click to reveal (try on your own first!)</summary>

```python
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("followup_visits.csv")
X = df.drop(columns=["appt_id", "missed"])
y = df["missed"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Train:", X_train.shape, " Test:", X_test.shape)
print("Missed rate, train:", round(y_train.mean() * 100, 1), "%  test:", round(y_test.mean() * 100, 1), "%")
```
</details>

In [ ]:
# Step 1: load the raw data and lock away a test set


✅ **Expected output**
```
Train: (576, 10)  Test: (144, 10)
Missed rate, train: 30.9 %  test: 31.2 %
```
💡 X is the capital-letter name for the feature table and y is the target column. random_state fixes the shuffle so everyone gets the same split. stratify=y keeps the share of missed appointments the same in both parts.

🆘 *If you see* `ValueError: not enough values to unpack` → train_test_split gives back four things, so you need four names on the left, in this order: X_train, X_test, y_train, y_test.

---
## Step 2 · Set the bar: the lazy baseline  *(~4 min)*
**Concept:** DummyClassifier, .fit, .score, .predict

📖 Before Maya celebrates any score, she asks: what would a lazy model get? One that says 'everyone will attend' and never phones anybody.

🎯 **Your task**
1. Create a `DummyClassifier` that always predicts the most common answer.
2. Fit it on the training set and score it on the test set.
3. Count how many missed appointments it predicts. Is this model any use to Maya?

<details><summary>👀 Need the code? Click to reveal (try on your own first!)</summary>

```python
from sklearn.dummy import DummyClassifier

lazy = DummyClassifier(strategy="most_frequent")
lazy.fit(X_train, y_train)
print("Lazy baseline:", round(lazy.score(X_test, y_test) * 100, 1), "% correct")
print("Missed appointments it predicts:", lazy.predict(X_test).sum(), "of", y_test.sum())
```
</details>

In [ ]:
# Step 2: the lazy baseline


✅ **Expected output**
```
Lazy baseline: 68.8 % correct
Missed appointments it predicts: 0 of 45
```
💡 The baseline is right about 7 times in 10 without learning anything, because most patients do attend. Any real model has to beat this number, or it has learned nothing useful.

🆘 *If you see* `NameError: name 'X_train' is not defined` → Run Step 1 first. Every step today uses its variables.

---
## Step 3 · A first real model: study, then exam  *(~5 min)*
**Concept:** KNeighborsClassifier, train score vs test score

📖 Now the 'similar patients' model from Tuesday, on three columns that have no gaps. Maya scores it twice: on the patients it studied, and on the hidden ones.

🎯 **Your task**
1. Fit a `KNeighborsClassifier` on the three simple columns of the training set.
2. Score it on the training set, then on the test set.
3. Which score is higher? Which one would you report to Kofi?

<details><summary>👀 Need the code? Click to reveal (try on your own first!)</summary>

```python
from sklearn.neighbors import KNeighborsClassifier

simple_cols = ["age", "days_waited", "prev_missed"]

model = KNeighborsClassifier()
model.fit(X_train[simple_cols], y_train)
print("Score on patients it studied:", round(model.score(X_train[simple_cols], y_train) * 100, 1), "%")
print("Score on hidden patients:    ", round(model.score(X_test[simple_cols], y_test) * 100, 1), "%")
```
</details>

In [ ]:
# Step 3: fit and score a first model


✅ **Expected output**
```
Score on patients it studied: 76.9 %
Score on hidden patients:     74.3 %
```
💡 The training score is like marking an exam made of the practice questions: it flatters the student. Only the score on unseen patients tells you how the model will do next month. A big gap between the two is called overfitting.

🆘 *If you see* `ValueError: Input X contains NaN` → You passed all the columns. For now use only X_train[simple_cols], which has no gaps.

---
## Step 4 · One split is one opinion  *(~5 min)*
**Concept:** random_state, variation between splits

📖 Kofi asks: 'If you had hidden different patients, would you get the same score?' Maya tries five different shuffles to find out.

🎯 **Your task**
1. Loop over five different `random_state` values.
2. Each time, split the whole table again, fit and score.
3. Look at the lowest and highest score. How far apart are they?

<details><summary>👀 Need the code? Click to reveal (try on your own first!)</summary>

```python
for seed in [1, 2, 3, 4, 5]:
    Xa, Xb, ya, yb = train_test_split(X[simple_cols], y, test_size=0.2, random_state=seed, stratify=y)
    model = KNeighborsClassifier()
    model.fit(Xa, ya)
    print("Shuffle", seed, "→", round(model.score(Xb, yb) * 100, 1), "%")
```
</details>

In [ ]:
# Step 4: try five different splits


✅ **Expected output**
```
Shuffle 1 → 70.8 %
Shuffle 2 → 68.8 %
Shuffle 3 → 74.3 %
Shuffle 4 → 65.3 %
Shuffle 5 → 69.4 %
```
💡 Same data, same model, and the score moves by several points depending on which patients landed in the test set. A single split is one opinion. We need a way to get several.

🆘 *If you see* `IndentationError: expected an indented block` → The four lines under the for line must each start with 4 spaces.

---
## Step 5 · Cross-validation: five exams, not one  *(~6 min)*
**Concept:** cross_val_score, cv=5, mean and standard deviation

📖 Cross-validation cuts the training set into 5 equal parts. Each part takes one turn as the exam while the other four are the study material. Five scores, one average.

🎯 **Your task**
1. Run `cross_val_score` with `cv=5` on the training set.
2. Print the five scores.
3. Print the mean and the standard deviation.

<details><summary>👀 Need the code? Click to reveal (try on your own first!)</summary>

```python
from sklearn.model_selection import cross_val_score

scores = cross_val_score(KNeighborsClassifier(), X_train[simple_cols], y_train, cv=5)
print((scores * 100).round(1))
print("Mean:", round(scores.mean() * 100, 1), "%  ± ", round(scores.std() * 100, 1))
```
</details>

In [ ]:
# Step 5: 5-fold cross-validation


✅ **Expected output**
```
[62.9 71.3 69.6 66.1 68.7]
Mean: 67.7 %  ±  2.9
```
💡 Every training patient is used for studying four times and for the exam exactly once. The mean is your honest estimate, and the ± tells you how much to trust it. The test set is still locked in the drawer.

🆘 *If you see* `TypeError: ... missing 1 required positional argument: 'y'` → cross_val_score needs three things: the model, the features and the target.

---
## Step 6 · Your first pipeline  *(~6 min)*
**Concept:** Pipeline, named steps

📖 Now for the columns with gaps. On Tuesday Maya filled and scaled by hand, using the whole table, which let the exam patients influence the preparation. A pipeline does the preparation inside each fold, so nothing leaks.

🎯 **Your task**
1. Build a `Pipeline` with three named steps: fill, scale, model.
2. Cross-validate it on the five number columns.
3. Compare the mean with Step 5. Did the two extra columns and the scaling help?

<details><summary>👀 Need the code? Click to reveal (try on your own first!)</summary>

```python
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

num_cols = ["age", "distance_km", "monthly_income", "days_waited", "prev_missed"]

num_pipe = Pipeline([
    ("fill", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("model", KNeighborsClassifier()),
])

scores = cross_val_score(num_pipe, X_train[num_cols], y_train, cv=5)
print("Number columns, in a pipeline:", round(scores.mean() * 100, 1), "%  ± ", round(scores.std() * 100, 1))
```
</details>

In [ ]:
# Step 6: fill → scale → model, in one pipeline


✅ **Expected output**
```
Number columns, in a pipeline: 74.3 %  ±  4.4
```
💡 A pipeline is a list of (name, tool) pairs. When it is fitted, each step learns from the study patients only and then passes its output to the next step. The last step is the model. You can use the whole pipeline as if it were one model.

🆘 *If you see* `TypeError: Pipeline.__init__() takes ...` → The steps go inside one list of pairs: Pipeline([("fill", ...), ("scale", ...), ("model", ...)]). Check your square brackets.

---
## Step 7 · Different columns, different treatment  *(~6 min)*
**Concept:** ColumnTransformer, OneHotEncoder(handle_unknown="ignore")

📖 Numbers need a median and a scaler. Words need the most common value and one-hot encoding. A ColumnTransformer is the triage desk that sends each column to the right treatment.

🎯 **Your task**
1. Build a small pipeline for the number columns and another for the category columns.
2. Join them with a `ColumnTransformer` called `prep`.
3. Fit it on the training set and compare the shape before and after.

<details><summary>👀 Need the code? Click to reveal (try on your own first!)</summary>

```python
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

cat_cols = ["sex", "ward", "transport", "has_phone", "sms_reminder"]

numbers = Pipeline([
    ("fill", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
categories = Pipeline([
    ("fill", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore")),
])
prep = ColumnTransformer([
    ("num", numbers, num_cols),
    ("cat", categories, cat_cols),
])

prepared = prep.fit_transform(X_train)
print("Before:", X_train.shape, " After:", prepared.shape)
print("Gaps left:", pd.isna(prepared).sum())
```
</details>

In [ ]:
# Step 7: one treatment per column type


✅ **Expected output**
```
Before: (576, 10)  After: (576, 18)
Gaps left: 0
```
💡 Each entry is (name, treatment, list of columns). The 5 category columns became 13 one-hot columns, which is why 10 columns turned into 18. handle_unknown="ignore" means a new, unseen category next month won't crash the model.

🆘 *If you see* `ValueError: A given column is not a column of the dataframe` → A name in num_cols or cat_cols is misspelled. They must match the CSV exactly.

---
## Step 8 · The full pipeline, cross-validated  *(~5 min)*
**Concept:** a pipeline inside a pipeline

📖 Preparation plus model, in one object. Raw data goes in at one end and predictions come out of the other. Maya cross-validates the whole thing.

🎯 **Your task**
1. Create a pipeline with two steps: `prep`, then the model.
2. Cross-validate it on the **full** training table, gaps and words included.
3. Compare with Step 6. What did the category columns add?

<details><summary>👀 Need the code? Click to reveal (try on your own first!)</summary>

```python
knn = Pipeline([
    ("prep", prep),
    ("model", KNeighborsClassifier()),
])

scores = cross_val_score(knn, X_train, y_train, cv=5)
print("All columns, KNN:", round(scores.mean() * 100, 1), "%  ± ", round(scores.std() * 100, 1))
```
</details>

In [ ]:
# Step 8: prep + model


✅ **Expected output**
```
All columns, KNN: 78.0 %  ±  0.9
```
💡 Nine steps from Tuesday now live in one object. In every fold, the medians, the scaling and the encoding are learned from that fold's study patients only, so the score is honest.

🆘 *If you see* `NameError: name 'prep' is not defined` → Run Step 7 first.

---
## Step 9 · A fair contest between models  *(~6 min)*
**Concept:** same preparation, same folds, different model

📖 Is 'similar patients' the best approach? Because the preparation is in a pipeline, Maya can swap the last step and give three models exactly the same exam.

🎯 **Your task**
1. Put three models in a dictionary.
2. Loop over them, building a pipeline with the same `prep` each time.
3. Which model has the best mean? Is it clearly ahead, given the ±?

<details><summary>👀 Need the code? Click to reveal (try on your own first!)</summary>

```python
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

models = {
    "KNN": KNeighborsClassifier(),
    "Logistic regression": LogisticRegression(max_iter=1000),
    "Decision tree": DecisionTreeClassifier(random_state=42),
}

for name, m in models.items():
    pipe = Pipeline([("prep", prep), ("model", m)])
    scores = cross_val_score(pipe, X_train, y_train, cv=5)
    print(name, "→", round(scores.mean() * 100, 1), "%  ± ", round(scores.std() * 100, 1))
```
</details>

In [ ]:
# Step 9: compare three models


✅ **Expected output**
```
KNN → 78.0 %  ±  0.9
Logistic regression → 79.7 %  ±  2.1
Decision tree → 75.2 %  ±  3.0
```
💡 Logistic regression draws a weighted score from the features, and a decision tree asks a chain of yes/no questions. You'll study both later in the programme. Today's point: cross-validation is how you choose between them fairly.

🆘 *If you see* `ConvergenceWarning` → A yellow warning, not an error. Check that you wrote max_iter=1000.

---
## Step 10 · Mini-project: open the drawer, then build Maya's call list  *(~8 min)*
**Concept:** final fit, test score, .predict_proba

📖 Maya has chosen her model. Now, and only now, she opens the locked drawer for one final exam. Then she uses the model on tomorrow's patients to decide whom to phone first.

🎯 **Your task**
1. Build the winning pipeline and fit it on the whole training set.
2. Score it once on the test set, and compare with the lazy baseline from Step 2.
3. Create tomorrow's three patients. Notice that one has gaps.
4. Add each patient's risk of missing with `predict_proba`, and sort the list.

<details><summary>👀 Need the code? Click to reveal (try on your own first!)</summary>

```python
best = Pipeline([("prep", prep), ("model", LogisticRegression(max_iter=1000))])
best.fit(X_train, y_train)
print("Final test score:", round(best.score(X_test, y_test) * 100, 1), "%")

tomorrow = pd.DataFrame({
    "age": [34, 6, 61],
    "sex": ["F", "M", "M"],
    "ward": ["Maternity", "Paediatrics", "Outpatients"],
    "transport": ["Bus", None, "Walk"],
    "distance_km": [4.0, None, 14.5],
    "monthly_income": [180, None, 90],
    "days_waited": [7, 30, 45],
    "prev_missed": [0, 1, 2],
    "has_phone": ["Yes", "Yes", "No"],
    "sms_reminder": ["Yes", "No", "No"],
})

risk = best.predict_proba(tomorrow)[:, 1]
call_list = tomorrow[["age", "ward", "days_waited", "prev_missed", "sms_reminder"]].copy()
call_list["risk_of_missing"] = risk.round(2)
print(call_list.sort_values("risk_of_missing", ascending=False))
```
</details>

In [ ]:
# Step 10: final exam and the call list


✅ **Expected output**
```
Final test score: 78.5 %
   age         ward  days_waited  prev_missed sms_reminder  risk_of_missing
2   61  Outpatients           45            2           No             0.99
1    6  Paediatrics           30            1           No             0.37
0   34    Maternity            7            0          Yes             0.00
```
💡 An honest process gives a test score in the same range as the cross-validation estimate. predict_proba returns two columns, the chance of 0 and the chance of 1, and [:, 1] keeps the chance of missing. The pipeline filled the second patient's gaps on its own.

🆘 *If you see* `ValueError: columns are missing` → tomorrow must have the same column names as X. Check your spelling against X.columns.

---
# 🌟 Finished early? Challenges
Optional extras for fast finishers.

### Challenge 1 · Ten folds
Cross-validate the `knn` pipeline with `cv=10`. Print the mean and the standard deviation. How do they compare with 5 folds?

In [ ]:
# Challenge 1


### Challenge 2 · How many neighbours? (a taste of next week)
Cross-validate the `knn` pipeline with 3, 5, 15 and 25 neighbours. Use `knn.set_params(model__n_neighbors=k)` inside a loop. Which value is best?

In [ ]:
# Challenge 2


### Challenge 3 · Accuracy isn't everything
Of the patients in the test set who really missed, how many did `best` catch? Use `confusion_matrix` from `sklearn.metrics`. Rows are the truth and columns are the prediction.

In [ ]:
# Challenge 3


---
## 🎉 Well done!
You set a baseline, cross-validated, built a leak-free pipeline, compared models fairly and produced a call list. **Save this notebook** (Ctrl + S), then commit it and push it to GitHub.

Next: the **Week 3 quiz** and **Week3_Exercises_CV_Pipelines.ipynb**.